# 03 - Where should the sensors go?

*Idealised synthetic experiment. Nothing here is an estimate for any real place.*

Notebook 02 showed that a few sensor readings can reveal how much each source emits, **if the sensors are in useful
places**. Real networks are small (money, power, people), so *where* we put a handful of sensors matters a lot.

**What is compared.** For a fixed number of sensors (the *budget*) we compare three ways to choose their positions
from a grid of allowed sites:

* **random**: sites picked at random (we draw many random layouts to see the whole range of luck);
* **spread**: sites spread out as evenly as possible over the domain (a sensible rule of thumb, ignores the physics);
* **greedy**: add one site at a time, always the site that most reduces the average reconstruction error on a set of
  *design* scenarios. This uses the physics (the model) but is a heuristic: *greedy is not guaranteed to be optimal*.

**How we keep it honest.**

1. Layouts are chosen using *design* scenarios only (random sources strengths and winds, with their own noise).
2. They are then scored on *independent test* scenarios (different strengths and winds, fresh noise) that the
   selection never saw. This is called *held-out* evaluation. We also print the design-stage score so any
   over-fitting to the design scenarios is visible.
3. We always show the spread of results (mean and a band of $\pm 1$ standard deviation), not just one number.

**Caveats, up front.** The numbers of scenarios and noise draws are modest (so that this notebook runs in a few
minutes), the world is synthetic, the inversion knows the model exactly in the matched experiments, and no method is
promised to win on every single scenario. Treat the results as an illustration of the method, not a verdict.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from pravah.config import load_config, scenario_from_dict
from pravah.forward import simulate
from pravah.observations import ObservationSpec, observe
from pravah.inverse import ResponseCache, InferenceOptions, fit_sources, reconstruct_linear
from pravah.metrics import population_weight
from pravah import experiments as ex
from pravah import plotting

pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:.4g}")

## 1. The experiment settings

We start from `configs/smoke.json` (a coarse, fast version of the benchmark) and enlarge it modestly:
a 6 x 6 grid of allowed sensor sites, budgets of 1, 2, 3 and 5 sensors, 3 design and 3 test scenarios, 2 noise draws
each, and 10 random layouts per budget. The full study (`configs/benchmark.json`) uses a finer grid and more of
everything; run it from the command line if you want tighter numbers.

All random draws have explicit seeds, so re-running gives identical results. The design seed and the test seed are
different, and the code refuses to run if the two sets of scenarios overlap.

In [ ]:
cfg = load_config("../configs/smoke.json")
cfg["design"].update(candidates_n=6, budgets=[1, 2, 3, 5], n_design=3, n_test=3, noise_reps=2, n_random=10,
                     weighted=True, heldout_n=5)
cfg["mismatch"]["poor_layout_sites"] = [[833.0, 9167.0], [2500.0, 9167.0], [833.0, 7500.0]]  # top-left corner
d = cfg["design"]
base = scenario_from_dict(cfg["scenario"])
print("seeds: design", d["design_seed"], " test", d["test_seed"], "(must differ)")
print("noise std", cfg["inverse"]["noise_std"], "ug/m^3 on every reading")

specs = ex.make_ensemble(base, d["n_test"], d["test_seed"], cfg["q"], "t")
for s in specs:
    w = ", ".join(f"({x.u:.2f}, {x.v:.2f}) m/s from t={x.t_start:.0f} s" for x in s.scenario.transport.wind)
    print(s.name, "q =", [f"{q:.3g}" for q in s.q], "wind:", w)

Each test scenario has its own source strengths (0.5x to 1.5x the nominal ones) and its own wind (speed
0.6 to 1.6 m/s, direction within 40 degrees of west-to-east; every second scenario has a wind change part-way
through). The inversion is told the wind that really blew (the "matched" experiment), so the only unknowns are the
source strengths.

## 2. Run the comparison

This does the following (all inside `run_benchmark`): builds the design and test scenarios (one forward simulation
each, then cheap linear algebra for every layout), selects the greedy layouts from the design scenarios only,
and evaluates every layout on the test scenarios. We also select a second greedy layout
(`greedy_weighted`) whose objective weights errors by a synthetic "population" map. That is explained in section 8.

In [ ]:
t0 = time.perf_counter()
res = ex.run_benchmark(cfg, out_dir=None, progress=print)
print(f"done in {time.perf_counter() - t0:.0f} s; peak memory {res.peak_memory_mb:.0f} MB")
print("time per stage (s):", {k: round(v, 1) for k, v in res.timings.items()})

## 3. Error versus number of sensors

The line is the **mean** test RMSE (root-mean-square error of the reconstructed concentration field over the whole
map and time, in micrograms per cubic metre). The shaded band is **mean $\pm$ 1 standard deviation of the individual
results** across layouts, test scenarios and noise draws. It describes how much results vary, it is not a confidence
interval. The vertical axis is logarithmic: badly placed sensors can be worse by many orders of magnitude. If no
sensor sees a plume, the strength of that source is simply not constrained by the data (we use no regularising prior
on purpose, so the fit is free to return an absurd value; the tables below are therefore dominated by a few extreme
draws, which is why we also show medians).

In [ ]:
fig = ex.plot_error_bands(res.summary, "uniform")
plt.show()

In [ ]:
tab = res.summary[res.summary["weighting"] == "uniform"].pivot(index="method", columns="budget", values="mean_rmse")
print("Mean test RMSE (ug/m^3) by method and number of sensors")
tab


In [ ]:
tab = res.summary[res.summary["weighting"] == "uniform"].pivot(index="method", columns="budget", values="median_rmse")
print("Median test RMSE: robust to the rare very bad draws")
tab


**Reading the table.** Random layouts vary hugely: a few of them are very bad and drag the mean far above the
median. Spread layouts are sensible but physics-blind. Greedy uses the model, so it tends to put sensors where
the plumes go.

### Paired comparison

A fairer way to compare two methods is to look at the *difference* on the same scenario and noise draw. Negative
numbers mean the first method has the smaller error. `frac_a_better` is the fraction of paired draws in which the
first method won; `sd_diff` is the standard deviation of the differences. With only a handful of scenarios these are
descriptive, not significance tests.

In [ ]:
p = res.paired[res.paired["weighting"] == "uniform"]
p[p["comparison"].isin(["greedy - spread", "greedy - random_mean", "spread - random_mean"])]

## 4. Where did greedy put the sensors?

Blue stars are the sources (nominal positions), grey dots are the allowed sites, red dots the chosen sites. For
greedy the numbers give the order in which sites were added. Notice how the greedy sites relate to where the
plumes travel (west to east, slightly north).

In [ ]:
b = max(d["budgets"])
fig = ex.plot_selected_sites(res, b)
plt.show()

## 5. Reconstruction errors and held-out performance

Left: the error of the reconstructed concentration *everywhere*. Right: the error at 25 extra sites that were never
candidates and are never used in any fit, a "held-out sensor" check. If a layout were good only at its own sites,
the right panel would look much worse than the left one.

In [ ]:
fig = ex.plot_heldout_vs_regional(res.summary)
plt.show()

### Design stage versus test stage

`design_rmse` is the score on the scenarios used for selection, `pooled_rmse` is on the independent test scenarios,
and `gap_rmse = pooled_rmse - design_rmse`. A large positive gap means the layout was tuned to the design scenarios
and does not carry over (over-fitting). Random layouts are never tuned, their design score is shown for reference.

In [ ]:
s = res.summary[(res.summary["weighting"] == "uniform") & res.summary["method"].isin(["greedy", "spread"])]
s[["method", "budget", "design_rmse", "pooled_rmse", "gap_rmse", "mean_heldout_rmse", "n_rows"]]

### One scenario in pictures

We take the first test scenario, give 3 sensors to each of the greedy and spread layouts, add noise, fit the
source strengths and rebuild the whole concentration map. Panels show truth, estimate and estimate minus truth.

In [ ]:
cache = ResponseCache()
spec = specs[0]
truth = simulate(spec.scenario, np.array(spec.q)).concentration
obs_times = [float(t) for t in np.arange(600.0, base.times.duration + 1e-9, 600.0)]
for method in ("greedy", "spread"):
    lay = res.candidates.sensor_layout(res.layouts[method][3][0])
    obs = observe(truth, lay, ObservationSpec(obs_times, cfg["inverse"]["noise_std"], seed=123))
    fit = fit_sources(spec.scenario, obs, InferenceOptions(), cache)
    est = reconstruct_linear(cache.get(spec.scenario), fit)
    err = float(np.sqrt(((est - truth) ** 2).mean()))
    print(f"{method}: sensors {lay.ids}, q_hat/q_true = {np.round(fit.q_hat / np.array(spec.q), 2)}, RMSE {err:.4f}")
    fig = plotting.plot_compare_fields(truth, est)
    fig.suptitle(f"{method} layout, 3 sensors")
    plt.show()

## 6. A deliberately poor layout, and identifiability

Now three sensors packed into the top-left corner, upwind of all the sources and away from the plumes. We keep the
benchmark layouts *fixed* and re-evaluate (see section 7). For each test scenario we also print identifiability
diagnostics of the 3-source fit:

* **singular values** of the (noise-whitened) sensitivity matrix: a value below 1 means that direction in
  source-strength space is not resolved above the noise;
* **effective rank**: how many singular values exceed 1 (3 would mean all three sources are determined);
* **condition number**: largest over smallest singular value (huge means near-degenerate);
* **q_std_rel**: the uncertainty of each strength relative to its true value (values far above 1 mean "unknown").

A fit can still return a *small* concentration error while the strengths are essentially undetermined; that is why
we look at both.

## 7. Model mismatch: when the world is not quite the model

Up to now the inversion knew the physics exactly and the truth was generated by the very same numerical model (an
"inverse crime"). Here the layouts stay **fixed** (as designed in the matched case) and we test them on the same
test scenarios and the same noise when the truth differs from the inversion model, one change at a time:

* **refined_truth**: the truth is simulated on a grid twice as fine, with smaller time steps; sensors read that fine
  field, and the error is measured against it averaged back onto the coarse grid;
* **wind_perturbed**: the true wind is 20% stronger and rotated by 15 degrees, but the inversion is given the
  *declared* wind;
* **extra_source**: the truth has a fourth source the inversion does not know about.

We report the paired difference (mismatch minus matched, same scenario, noise and layout). Positive means worse. We
do not expect every single draw to get worse.

In [ ]:
t0 = time.perf_counter()
mm = ex.run_mismatch(cfg, res.layouts_json(d["candidates_n"]), out_dir=None, cache=cache, progress=print)
print(f"done in {time.perf_counter() - t0:.0f} s")
fig = ex.plot_mismatch_rmse(mm.summary)
plt.show()

In [ ]:
fig = ex.plot_mismatch_differences(mm.paired)
plt.show()
pm = mm.paired[mm.paired["method"].isin(["greedy", "spread"])]
pm[["condition", "method", "budget", "mean_diff_rmse", "sd_diff_rmse", "n", "frac_worse"]]

In [ ]:
print("Poor layout versus greedy and spread at the same budget (matched conditions, paired differences):")
display(mm.poor_paired)
diag = mm.poor_diagnostics
diag[diag["case"] == "t0"][["method", "case", "rep", "rank", "effective_rank", "cond_number", "sv_min", "sv_max",
                            "q_std_rel_S1", "q_std_rel_S2", "q_std_rel_S3", "rmse"]]

In [ ]:
fig = ex.plot_poor_layout(mm.poor_diagnostics, mm.summary, res.candidates, mm.layouts, base)
plt.show()

**What to take away.** Layouts that look good in the matched world can lose accuracy under mismatch,
and the extra-source case adds error the inversion cannot explain (it has nowhere to put the unknown emission, so
it distorts the three strengths it does model). The poor layout is bad in every respect: its sensors see almost
nothing, so the effective rank collapses and the strengths are not determined.

## 8. Optional: spatial weighting (this changes the *score*, not the physics)

Perhaps errors matter more where people live. We invent a synthetic "population" map (three blobs plus a small floor;
**not real data**) and use it as an importance weight in the error metric: the mean squared error becomes a weighted
average over the map. The PDE and the data are untouched. We then ask for greedy sites that minimise this weighted
error (`greedy_weighted`), and evaluate every layout with *both* the unweighted and the weighted metric (column
`weighting`: `uniform` or `population`).

In [ ]:
w = population_weight(base.domain)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.3))
im = axes[0].imshow(w, origin="lower", extent=(0, base.domain.lx, 0, base.domain.ly))
fig.colorbar(im, ax=axes[0], label="importance weight (mean 1)")
axes[0].set_title("synthetic population-like weight")
c = res.candidates
for m, col in (("greedy", "tab:red"), ("greedy_weighted", "tab:orange")):
    lay = list(res.layouts[m][b][0])
    axes[1].plot(c.x[lay], c.y[lay], "o", color=col, ms=9, label=m, alpha=0.7)
axes[1].plot(c.x, c.y, ".", c="0.75")
axes[1].plot([s.x for s in base.sources], [s.y for s in base.sources], "b*", ms=12, label="sources")
axes[1].set(xlim=(0, base.domain.lx), ylim=(0, base.domain.ly), aspect="equal", title=f"{b}-sensor greedy layouts")
axes[1].legend(fontsize=8)
plt.show()
s = res.summary[res.summary["method"].isin(["greedy", "greedy_weighted"]) & (res.summary["budget"] == b)]
s[["method", "weighting", "budget", "mean_rmse", "sd_rmse", "design_rmse", "pooled_rmse"]]

If the two greedy layouts coincide at some budget, the weights did not change the choice there, which is a
perfectly valid outcome.

### Optional: try other settings interactively

The cell below needs `ipywidgets` and only runs when you press the button. Everything above works without it.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    noise = widgets.FloatSlider(value=cfg["inverse"]["noise_std"], min=0.001, max=0.02, step=0.001,
                                description="noise std")
    btn = widgets.Button(description="Run (about 1 min)")
    out = widgets.Output()

    def go(_):
        with out:
            clear_output()
            c2 = {**cfg, "inverse": {**cfg["inverse"], "noise_std": noise.value},
                  "design": {**cfg["design"], "budgets": [1, 3], "n_random": 5, "weighted": False}}
            r2 = ex.run_benchmark(c2, cache=cache)
            ex.plot_error_bands(r2.summary, "uniform")
            plt.show()

    btn.on_click(go)
    display(widgets.VBox([noise, btn, out]))
except Exception as exc:  # widgets are optional
    print("widgets not available:", exc)

## Summary and limits

* Compare greedy, spread and random in the tables above: the paired table (`frac_a_better`) shows how often one
  method beats another on the same scenario and noise; read it rather than assuming greedy always wins.
* Layouts were tuned on design scenarios and judged on independent test scenarios; compare `design_rmse` to
  `pooled_rmse`.
* Under model mismatch every method degrades by some amount; the changes are descriptive and need not appear in
  every draw.
* Limits: modest numbers of scenarios and noise draws, synthetic physics, a coarse grid here, greedy selection is a
  heuristic and not the optimum, and no result is claimed for any real location.